# Jersey digit detector (YOLO26s)

Trains a detector for single digits 0-9 on player crops, on the zip built by `scripts/build_jersey_dataset.py`.
It replaces the capstone model, which was trained on mislabelled classes (balls, whole players and whole-number
boxes merged in as the digits 0-2) and with horizontal flips (a mirrored digit is a different digit, and a
mirrored two-digit number reads backwards). Details: `docs/results/actions.md`.

- Data: the two Roboflow sets with digit classes, split **by match**: test = Japan-Poland, Thailand and
  Poland-Italy (teams in no training match), valid = two other matches, train = the rest (plus baseball and
  football digit crops, train only).
- No horizontal flips (`fliplr=0`).

Runs on **Colab** or **Kaggle** (detected automatically):
- Colab: Runtime -> Change runtime type -> T4 GPU, then Run all. Data and checkpoints live on Drive
  (`MyDrive/volleyball/...`); after a disconnect, Run all again resumes from `last.pt`.
- Kaggle: attach the private dataset `jersey-digits`, enable GPU and Internet, then Save Version -> Run All
  (or `kaggle kernels push`). Outputs land in `/kaggle/working`.


In [ ]:
import os, glob, shutil
KAGGLE = os.path.exists('/kaggle/input')
if KAGGLE:
    RUNS, MODELS = '/kaggle/working/runs', '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS, MODELS = '/content/drive/MyDrive/volleyball/runs', '/content/drive/MyDrive/volleyball/models'
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()


In [ ]:
# Unpack the dataset into a writable folder (ultralytics writes label caches next to the labels)
DATA = '/kaggle/working/data' if KAGGLE else '/content/data'
shutil.rmtree(DATA, ignore_errors=True); os.makedirs(DATA)
if KAGGLE:
    found = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
    if found:  # Kaggle unpacked the zip on upload
        shutil.copytree(os.path.dirname(found[0]), f'{DATA}/jersey_digits')
    else:
        shutil.unpack_archive(glob.glob('/kaggle/input/**/jersey_digits.zip', recursive=True)[0], DATA)
else:
    shutil.unpack_archive('/content/drive/MyDrive/volleyball/datasets/jersey_digits.zip', DATA)
print(glob.glob(f'{DATA}/**/data.yaml', recursive=True))


In [ ]:
import yaml
cfg = sorted(glob.glob(f'{DATA}/**/data.yaml', recursive=True))[0]
root = os.path.dirname(cfg)
d = yaml.safe_load(open(cfg))
d['path'] = root
for split, name in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
    d[split] = os.path.join(root, name, 'images')
yaml.safe_dump(d, open(cfg, 'w'))
print(cfg); print({k: len(os.listdir(d[k])) for k in ('train', 'val', 'test')})


In [ ]:
from ultralytics import YOLO
import torch
RUN_NAME = 'jersey_digits_v1'
last = f'{RUNS}/{RUN_NAME}/weights/last.pt'
# last.pt has epoch == -1 once training finished (or stopped early): then skip, don't retrain.
finished = os.path.exists(last) and torch.load(last, map_location='cpu', weights_only=False).get('epoch') == -1

if finished:
    print('training already finished:', last)
elif os.path.exists(last):
    print('resuming from', last)
    model = YOLO(last)
    model.train(resume=True)
else:
    # crops are 92-184 px wide, so 320 px input is enough
    model = YOLO('yolo26s.pt')
    model.train(data=cfg, epochs=150, imgsz=320, batch=64, patience=40,
                fliplr=0.0,           # a mirrored digit is a different digit
                degrees=5, mosaic=0.5,
                project=RUNS, name=RUN_NAME, exist_ok=True, save_period=10)


In [ ]:
import json
model = YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt')
# test = matches whose teams are in no training match
m = model.val(data=cfg, split='test', imgsz=320)
res = {'test_map50': float(m.box.map50), 'test_map50_95': float(m.box.map), 'precision': float(m.box.mp),
       'recall': float(m.box.mr), 'ap50_per_digit': {str(int(c)): float(m.box.ap50[i]) for i, c in enumerate(m.box.ap_class_index)}}
print(json.dumps(res, indent=1))
os.makedirs(MODELS, exist_ok=True)
shutil.copy(f'{RUNS}/{RUN_NAME}/weights/best.pt', f'{MODELS}/jersey_digits_yolo26s.pt')
shutil.copy(f'{RUNS}/{RUN_NAME}/results.png', f'{MODELS}/jersey_digits_v1_results.png')
json.dump(res, open(f'{MODELS}/jersey_digits_v1_test.json', 'w'), indent=1)
print('saved to', MODELS)
